In [5]:
import pandas as pd

import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, chi2
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.pipeline import Pipeline
import pickle
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/peiyuwang/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [52]:
stemmer = PorterStemmer()
words = stopwords.words("english")
df = pd.read_csv('stress analysis.csv')
df['cleaned_subreddit'] = df['subreddit'].apply(lambda x: " ".join([stemmer.stem(i) for i in re.sub("[^a-zA-Z]", " ", x).split() if i not in words]).lower())

df.head()




,label,subreddit,cleaned_subreddit
0,stress,I man the front desk and my title is HR Custom...,i man front desk titl hr custom servic repres ...
1,stress,"More specifically, for example, I live with ro...",more specif exampl i live roommat i rememb las...
2,stress,"I have a lot of self esteem. I value myself, I...",i lot self esteem i valu i believ i smart good...
3,stress,"If I go to an interview for example, I'll know...",if i go interview exampl i know i good candid ...
4,stress,Like sleep would never be a simple thing for m...,like sleep would never simpl thing so recent i...


In [54]:
vectorizer = TfidfVectorizer(min_df= 3, stop_words="english", sublinear_tf=True, norm='l2', ngram_range=(1, 2))
final_features = vectorizer.fit_transform(df['cleaned_subreddit']).toarray()
final_features.shape


(430, 1197)

In [64]:
X = df['cleaned_subreddit']
Y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25)

pipeline = Pipeline([('vect', vectorizer),
                     ('chi',  SelectKBest(chi2, k='all')),
                     ('clf', LogisticRegression(random_state=0))])

model = pipeline.fit(X_train, y_train)
with open('LogisticRegression.pickle', 'wb') as f:
    pickle.dump(model, f)

ytest = np.array(y_test)

# confusion matrix and classification report(precision, recall, F1-score)
print(classification_report(ytest, model.predict(X_test)))
print(confusion_matrix(ytest, model.predict(X_test)))

              precision    recall  f1-score   support

      stress       1.00      0.23      0.37        44
      trauma       0.65      1.00      0.79        64

    accuracy                           0.69       108
   macro avg       0.83      0.61      0.58       108
weighted avg       0.79      0.69      0.62       108

[[10 34]
 [ 0 64]]


### Claude 3.5

In [7]:
def classify_text(text):
    """
    Classify text as stress, trauma, or unknown based on keyword matching.
    
    Args:
        text (str): Text to classify
    
    Returns:
        str: Classification category (stress, trauma, or unknown)
    """
    # Convert to lowercase for case-insensitive matching
    if not isinstance(text, str):
        return 'unknown'
    
    text_lower = text.lower()
    
    # Keywords for stress-related content
    stress_keywords = [
        'anxiety', 'nervous', 'worry', 'panic', 'overwhelmed', 'stress', 
        'pressure', 'tension', 'fear', 'dread', 'restless', 'uneasy', 
        'overthinking', 'social anxiety', 'performance anxiety', 
        'mental health', 'burnout', 'mental exhaustion'
    ]
    
    # Keywords for trauma-related content
    trauma_keywords = [
        'abuse', 'ptsd', 'trauma', 'assault', 'rape', 'harassment', 
        'violence', 'childhood trauma', 'sexual abuse', 'emotional abuse', 
        'domestic violence', 'survivor', 'triggering', 'flashback',
        'sexual assault', 'physical abuse', 'mental abuse'
    ]
    
    # Count keyword occurrences
    stress_count = sum(keyword in text_lower for keyword in stress_keywords)
    trauma_count = sum(keyword in text_lower for keyword in trauma_keywords)
    
    # Classify based on keyword counts
    if trauma_count > stress_count and trauma_count > 0:
        return 'trauma'
    elif stress_count > trauma_count and stress_count > 0:
        return 'stress'
    
    return 'unknown'

def classify_subreddits(input_file, output_file):
    """
    Classify subreddit entries from input CSV and save to output CSV.
    
    Args:
        input_file (str): Path to input CSV file
        output_file (str): Path to output CSV file
    """
    # Read the input CSV
    df = pd.read_csv(input_file)
    
    # Add classification column
    df['category'] = df['subreddit'].apply(classify_text)
    
    # Save to output CSV
    df.to_csv(output_file, index=False)
    
    # Print classification summary
    category_counts = df['category'].value_counts()
    print("Classification Summary:")
    print(category_counts)
    
    # Print a few example entries for each category
    print("\nExample Entries:")
    for category in ['stress', 'trauma', 'unknown']:
        print(f"\n{category.capitalize()} Examples:")
        print(df[df['category'] == category]['subreddit'].head(3).to_string())

input_file = 'stress analysis.csv'
output_file = 'classified_subreddits.csv'
classify_subreddits(input_file, output_file)
print(f"\nClassification complete. Results saved to {output_file}")

Classification Summary:
category
unknown    225
stress     124
trauma      81
Name: count, dtype: int64

Example Entries:

Stress Examples:
1    More specifically, for example, I live with ro...
2    I have a lot of self esteem. I value myself, I...
3    If I go to an interview for example, I'll know...

Trauma Examples:
165    Yesterday afternoon, two black males attacked ...
166    PTSD is life changing for the worse. Every day...
167    He‚Äôs abused my dog in the past and I‚Äôve no...

Unknown Examples:
0    I man the front desk and my title is HR Custom...
4    Like sleep would never be a simple thing for m...
6    Manageable enough to come home on weekends and...

Classification complete. Results saved to classified_subreddits.csv
